# Fare anomaly detector: evaluation

The detector (`backend/app/modules/ingestion/anomalies.py`) is an Isolation Forest fitted on
every import over route-relative fare and duration. Here it is measured against **known** errors:
the demo generator injects fares at 8x their real value, and a clean run of the same seed gives
the true fares, so precision and recall are exact. Runs locally in the backend environment.

In [1]:
import csv, io, json, sys, tempfile, zipfile, datetime
from datetime import date
from pathlib import Path
from app.core.config import get_settings
from app.datagen.generator import GeneratorOptions, generate
from app.modules.ingestion.anomalies import MIN_RELATIVE_JUMP, score_fares
from app.modules.ingestion.mapping import load_mapping
from app.modules.ingestion.pipeline import prepare
from app.modules.ingestion.rules import load_rules, load_vocabulary
from app.modules.ingestion.sources import open_source

In [2]:
tmp = Path(tempfile.mkdtemp())
results = []
settings = get_settings()
for seed in (7, 11, 23):
    opts = dict(seed=seed, start=date(2026, 11, 13), days=45)
    generate(tmp / "dirty.zip", GeneratorOptions(**opts)); generate(tmp / "clean.zip", GeneratorOptions(**opts, dirty_rate=0))
    rows = lambda z: list(csv.DictReader(io.StringIO(zipfile.ZipFile(z).read("flights.csv").decode())))
    spiked = {f"{d['flight_no'].replace(' ', '')}|{d['departure']}|{d['cabin']}"
              for c, d in zip(rows(tmp / "clean.zip"), rows(tmp / "dirty.zip"))
              if c["fare"] and d["fare"] and float(d["fare"]) > 4 * float(c["fare"])}
    prepared = prepare(open_source(tmp / "dirty.zip", max_members=10, max_uncompressed_mb=900),
                       load_mapping(settings.config_dir, "synthetic-v1"), load_rules(settings.config_dir),
                       load_vocabulary(settings.config_dir), catalog_airports=lambda: [])
    f = prepared.flights
    key = f.flight_number + "|" + f.departure_local.dt.strftime("%Y-%m-%d %H:%M") + "|" + f.cabin_class.str.capitalize()
    flagged, truth = score_fares(f), key.isin(spiked)
    tp, fp, fn = int((flagged & truth).sum()), int((flagged & ~truth).sum()), int((~flagged & truth).sum())
    results.append({"seed": seed, "fares": len(f), "injected": int(truth.sum()), "tp": tp, "fp": fp, "fn": fn})
    print(results[-1])

{'seed': 7, 'fares': 23769, 'injected': 12, 'tp': 12, 'fp': 0, 'fn': 0}


{'seed': 11, 'fares': 23895, 'injected': 12, 'tp': 12, 'fp': 0, 'fn': 0}


{'seed': 23, 'fares': 23739, 'injected': 12, 'tp': 12, 'fp': 0, 'fn': 0}


In [3]:
tp, fp, fn = (sum(r[k] for r in results) for k in ("tp", "fp", "fn"))
metrics = {"precision": tp / max(tp + fp, 1), "recall": tp / max(tp + fn, 1), "fares_checked": float(sum(r["fares"] for r in results)),
           "min_relative_jump": MIN_RELATIVE_JUMP}
print(metrics)
card = {"name": "price_anomaly", "version": datetime.date.today().strftime("%Y.%m.%d"),
        "trained_at": datetime.datetime.now(datetime.timezone.utc).isoformat(), "framework": "scikit-learn",
        "framework_version": __import__("sklearn").__version__, "dataset": "TripMind synthetic data, 3 seeds, injected 8x fares",
        "features": ["relative_log_fare", "relative_log_duration"], "metrics": metrics,
        "notes": "Fitted per import; this card records the evaluation, there is no stored artifact."}
Path("artifacts").mkdir(exist_ok=True); Path("artifacts/model_card.json").write_text(json.dumps(card, indent=2))

{'precision': 1.0, 'recall': 1.0, 'fares_checked': 71403.0, 'min_relative_jump': 1.0}


529